In [ ]:
# NLGCL Kaggle Notebook
# This notebook runs the NLGCL model on the Kaggle platform.
# It automatically sets up the environment, downloads the dataset, and runs training.

import os
import sys
import shutil

# 1. Setup Environment & Clone
# Ensure we are in the correct directory
if not os.path.exists('NLGCL') and not os.path.exists('recbole_gnn'):
    !git clone https://github.com/Jinfeng-Xu/NLGCL.git
    os.chdir('NLGCL')
    print("Repository cloned and entered.")
elif os.path.basename(os.getcwd()) != 'NLGCL' and os.path.exists('NLGCL'):
    os.chdir('NLGCL')
    print("Entered NLGCL directory.")
else:
    print(f"Current directory: {os.getcwd()}")

# Add current directory to sys.path
sys.path.append(os.getcwd())

# 2. Install Dependencies
# We check and install dependencies first to avoid runtime errors later.
try:
    import recbole
    print("RecBole is already installed.")
except ImportError:
    print("RecBole not found. Installing dependencies (this may take a few minutes)...")
    # Install main requirements
    !pip install -r requirements.txt
    # Install PyG dependencies (compatible with standard Kaggle/Colab environments)
    !pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
    print("Dependencies installed.")

# 3. Dataset Preparation
dataset_name = 'Yelp'
target_dir = f'dataset/{dataset_name}'

# Check if dataset is already in /kaggle/input (if user uploaded it)
if not os.path.exists(target_dir) and os.path.exists('/kaggle/input'):
    print("Searching for dataset in /kaggle/input...")
    found_in_input = False
    for root, dirs, files in os.walk('/kaggle/input'):
        # Look for RecBole atomic file or the folder itself
        if f'{dataset_name.lower()}.inter' in files or dataset_name in root.split(os.sep):
            # If we found the atomic file, the root is the dataset folder
            source = root
            print(f"Found dataset source at {source}. Linking/Copying to {target_dir}...")
            shutil.copytree(source, target_dir, dirs_exist_ok=True)
            found_in_input = True
            break
    if not found_in_input:
        print("Dataset not found in /kaggle/input.")

if not os.path.exists(target_dir):
    print(f"Dataset '{dataset_name}' not found locally. Downloading from RecBole S3...")
    import requests
    import zipfile
    
    # RecBole S3 URL for Yelp
    url = "https://recbole.s3-accelerate.amazonaws.com/ProcessedDatasets/Yelp/yelp.zip"
    
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)
        
    zip_path = os.path.join(target_dir, f"{dataset_name}.zip")
    
    try:
        print(f"Downloading from {url}...")
        response = requests.get(url, stream=True)
        response.raise_for_status()
        
        total_size = int(response.headers.get('content-length', 0))
        block_size = 1024 * 1024 # 1MB
        downloaded = 0
        
        with open(zip_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if downloaded % (50 * block_size) < 8192:
                         print(f"Downloaded {downloaded / block_size:.1f} MB...", end='\r')
                    
        print(f"\nDownload complete. Extracting...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(target_dir)
        
        # Cleanup
        if os.path.exists(zip_path):
            os.remove(zip_path)
            
        print(f"Dataset ready at {target_dir}")
        
    except Exception as e:
        print(f"Download failed: {e}")
else:
    print(f"Dataset '{dataset_name}' is ready at {target_dir}")

# 4. Run Training
from logging import getLogger
from recbole.utils import init_logger, init_seed, set_color
from recbole_gnn.config import Config
from recbole_gnn.utils import create_dataset, data_preparation, get_model, get_trainer
import torch

def run_single_model(args):
    # configurations initialization
    config = Config(
        model=args.model,
        dataset=args.dataset,
        config_file_list=args.config_file_list
    )
    
    # Ensure data_path is correctly set to local dataset/ folder
    if 'data_path' not in config:
        config['data_path'] = 'dataset/'

    init_seed(config['seed'], config['reproducibility'])
    init_logger(config)
    logger = getLogger()
    logger.info(config)

    # dataset filtering
    dataset = create_dataset(config)
    logger.info(dataset)

    # dataset splitting
    train_data, valid_data, test_data = data_preparation(config, dataset)

    # model loading and initialization
    # Note: Ensure the model class is available in recbole_gnn/model/general_recommender/nlgcl.py
    model = get_model(config['model'])(config, train_data.dataset).to(config['device'])
    logger.info(model)

    # trainer loading and initialization
    trainer = get_trainer(config['MODEL_TYPE'], config['model'])(config, model)

    # model training
    best_valid_score, best_valid_result = trainer.fit(
        train_data, valid_data, saved=True, show_progress=config['show_progress']
    )

    # model evaluation
    test_result = trainer.evaluate(test_data, load_best_model=True, show_progress=config['show_progress'])

    logger.info(set_color('best valid ', 'yellow') + f': {best_valid_result}')
    logger.info(set_color('test result', 'yellow') + f': {test_result}')


# Simulation of command-line arguments
class Args:
    def __init__(self):
        self.model = 'NLGCL'
        self.dataset = 'Yelp'
        self.config_files = ''
        self.config_file_list = ['properties/overall.yaml']

args = Args()

if torch.cuda.is_available():
    print(f"Running on GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU. Training might be slow.")

try:
    if os.path.exists(f'dataset/{args.dataset}'):
         run_single_model(args)
    else:
         print(f"Error: Dataset {args.dataset} setup failed. Please check logs above.")
except Exception as e:
    import traceback
    traceback.print_exc()